In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


In [84]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df = pd.read_csv("diabetes_binary_classification_data.csv")

print("Shape:", df.shape)
print("\nTarget distribution:")
print(df["Diabetes_binary"].value_counts(normalize=True).round(4))
print("\nColumn dtypes:")
print(df.dtypes.value_counts())
print("\nFirst rows:")
df.head()


Shape: (253680, 22)

Target distribution:
Diabetes_binary
0.0    0.8607
1.0    0.1393
Name: proportion, dtype: float64

Column dtypes:
float64    22
Name: count, dtype: int64

First rows:


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [85]:
missing = df.isna().mean().mul(100).sort_values(ascending=True)
missing = missing[missing > 0]

if not missing.empty:
    plt.figure(figsize=(8, 7))
    sns.barplot(x=missing.values, y=missing.index, color="steelblue")
    plt.xlabel("% missing")
    plt.title("Missing values per column")
    plt.show()
else:
    print("There are no missing values in the dataset.")


There are no missing values in the dataset.


Preprocessing and model setup

Binary 0/1 features are already encoded as yes/no categories and should stay numeric. Ordered categories such as Age, Education, Income and GenHlth remain numeric codes representing ordered groups, and the assignment specifically says not to force ordinary numeric correlation onto them.

We therefore keep the data as supplied, split it before fitting, and use a logistic regression classifier for the binary target.


In [ ]:
X = df.drop(columns=["Diabetes_binary"])
y = df["Diabetes_binary"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    stratify=y,
    random_state=42,
)


Training and evaluation

We fit a logistic regression model on the training split and evaluate it on the unseen test set. Because this is a class-imbalanced binary classification problem, we look at accuracy, precision, recall, F1, and the confusion matrix instead of regression metrics like RMSE or R².

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 score : {f1:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix:")
print(cm)

ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot(cmap="Blues")
plt.show()

Accuracy : 0.8621
Precision: 0.5166
Recall   : 0.1586
F1 score : 0.2427


Results

Confusion matrix from the logistic regression model:
[[42618  1049]
 [ 5948  1121]]

Accuracy : 0.8621
Precision: 0.5166
Recall   : 0.1586
F1 score : 0.2427

Interpretation:
- The model correctly identifies most class-0 cases.
- It misses many true class-1 cases, which is reflected in the low recall.
- This is expected in an imbalanced binary classification problem and is exactly why the assignment emphasizes recall and precision, not just accuracy.
